# PagedKV: 7B/8B evaluation on Colab Pro

Use a **Premium GPU** runtime. This FP16 protocol requires one GPU with at least **35 GiB** of device memory; an A100 40 GB qualifies. A T4 16 GB or L4 24 GB does not. Colab hardware and runtime duration are not guaranteed, so this notebook writes checkpoints directly to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Verify the assigned GPU

Do not continue unless this cell passes. If Colab assigns a smaller GPU, disconnect and request a Premium GPU again or use the Kaggle T4 x2 notebook.

In [ ]:
import subprocess

line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'
], text=True).strip().splitlines()[0]
gpu_name, memory_mib = [part.strip() for part in line.rsplit(',', 1)]
memory_gib = int(memory_mib) / 1024
print(f'GPU: {gpu_name}; memory: {memory_gib:.1f} GiB')
assert memory_gib >= 35, 'The FP16 7B/8B harness requires at least 35 GiB on one GPU.'

## Clone the pinned runner

The code revision is pinned because checkpoint resume validates source hashes. Do not pull a newer revision into an unfinished output directory.

In [ ]:
from pathlib import Path
import subprocess

PINNED_COMMIT = '2ade9a7'
repo = Path('/content/PagedKV')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JayGor-13/PagedKV.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', PINNED_COMMIT], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())

## Configure the model and persistent output

Qwen2.5-7B is public. For Llama-3.1-8B, change both values and provide `HF_TOKEN` through a private Colab secret after accepting Meta's model terms. Run one model at a time.

In [ ]:
import shlex
from pathlib import Path

MODEL = 'Qwen/Qwen2.5-7B-Instruct'
RUN_TAG = 'qwen7b-a100'
OUTPUT_ROOT = '/content/drive/MyDrive/PagedKV-results'

# For gated Llama, uncomment and add an HF_TOKEN Colab secret:
# MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
# RUN_TAG = 'llama8b-a100'

lines = [
    f'export MODEL={shlex.quote(MODEL)}',
    f'export RUN_TAG={shlex.quote(RUN_TAG)}',
    f'export OUTPUT_ROOT={shlex.quote(OUTPUT_ROOT)}',
]
if MODEL.startswith('meta-llama/'):
    from google.colab import userdata
    lines.append(f'export HF_TOKEN={shlex.quote(userdata.get("HF_TOKEN"))}')
Path('/content/pagedkv-colab.env').write_text('\n'.join(lines) + '\n')
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
print('Model:', MODEL)
print('Persistent output:', OUTPUT_ROOT)

## Install isolated environments

The runtime filesystem is temporary, so repeat installation after Colab replaces the VM. Checkpoints remain in Drive.

In [ ]:
%%bash
set -euo pipefail
cd /content/PagedKV
bash scripts/setup_kaggle_t4.sh

In [ ]:
%%bash
set -euo pipefail
cd /content/PagedKV
bash scripts/test_kaggle_t4.sh

## Mandatory single-GPU smoke matrix

This validates every method on both benchmarks. Repeat the cell to resume the same Drive checkpoint.

In [ ]:
%%bash
set -euo pipefail
cd /content/PagedKV
source /content/pagedkv-colab.env
export CUDA_VISIBLE_DEVICES=0
PYTHONPATH= .envs/t4-baselines/bin/python -u -m experiments.kaggle_t4 run \
  --gpus 1 --models "$MODEL" --smoke \
  --out "$OUTPUT_ROOT/${RUN_TAG}-smoke"

In [ ]:
import json
from pathlib import Path

report = json.loads(Path(f'{OUTPUT_ROOT}/{RUN_TAG}-smoke/comparison.json').read_text())
failed = [(r['benchmark'], r['method'], r['status']) for r in report['rows'] if r['status'] != 'completed']
print('Incomplete or failed:', failed)
assert not failed

## Full LongBench v2: all 503 examples at a 12.5% capacity target

Repeat this exact cell after a disconnect. Resume requires the same source, packages, configuration, and GPU model. Colab does not guarantee the same GPU across sessions; the runner refuses to mix hardware silently.

In [ ]:
%%bash
set -euo pipefail
cd /content/PagedKV
source /content/pagedkv-colab.env
export CUDA_VISIBLE_DEVICES=0
PYTHONPATH= .envs/t4-baselines/bin/python -u -m experiments.kaggle_t4 run \
  --gpus 1 --models "$MODEL" \
  --benchmarks longbenchv2 \
  --methods full h2o snapkv quest arkvale rocketkv freekv ours \
  --samples 503 --selection stratified \
  --prompt-cap 16384 --max-new-tokens 128 --context-capacity-pct 12.5 \
  --out "$OUTPUT_ROOT/${RUN_TAG}-longbench-full503-b12p5"

## Inspect completion and scores

In [ ]:
import pandas as pd
from IPython.display import display

csv_path = f'{OUTPUT_ROOT}/{RUN_TAG}-longbench-full503-b12p5/comparison.csv'
results = pd.read_csv(csv_path)
display(results[['method', 'status', 'samples', 'expected', 'subset_accuracy_pct',
                 'elapsed_sample_seconds', 'elapsed_total_seconds']])
assert len(results) == 8
assert ((results.status == 'completed') & (results.samples == 503) & (results.expected == 503)).all()

The resulting accuracy is the full 503-example **LongBench v2** comparison at a 16K prompt cap. It is not LongBench v1 Avg, RULER 128K, MATH-500, PG-19 perplexity, or the H200 batch-8 systems benchmark; those require their separate H200 protocols.